# Import dos dados e bibliotecas

In [1]:
import pandas as pd
import numpy as np

import pyarrow

In [2]:
df_caminhao = pd.read_parquet('datasets/datasets_tratados/df_caminhao_limpo.parquet', engine="pyarrow")
# df_escavadeira = pd.read_parquet('datasets/datasets_tratados/df_escavadeira.parquet', engine="pyarrow")

# Tratamento dos dados para modelagem

In [3]:
# 1. Garantir ordenação estrita por equipamento e data
df_caminhao['Data_Evento'] = pd.to_datetime(df_caminhao['Data_Evento'])
df_caminhao = df_caminhao.sort_values(by=['TAG', 'Data_Evento']).reset_index(
    drop=True
)

# Criamos um array para armazenar os tempos em minutos
minutos_ate_falha = np.full(len(df_caminhao), np.nan)

# 2. Processar por TAG para isolar os equipamentos
for tag, group in df_caminhao.groupby('TAG'):
    indices_grupo = group.index

    # Datas onde Is_Dont_Go == 1
    datas_dont_go = group[group['Is_Dont_Go'] == 1]['Data_Evento'].values

    if len(datas_dont_go) == 0:
        continue  # Máquinas sem falha continuam como NaN

    # Forçamos a busca a olhar estritamente para o futuro (+1 segundo)
    # para que o próprio Don't Go não olhe para si mesmo
    momentos_busca = group['Data_Evento'] + pd.Timedelta(seconds=1)

    # Encontra o índice do próximo Don't Go no futuro
    indices_busca = np.searchsorted(datas_dont_go, momentos_busca.values)

    # Identifica quem não tem mais nenhuma falha dali para a frente
    sem_futuro = indices_busca >= len(datas_dont_go)
    indices_busca[sem_futuro] = len(datas_dont_go) - 1

    # Resgata a data do próximo Don't Go
    proximas_datas = datas_dont_go[indices_busca]

    # Calcula a diferença de tempo absoluta
    diferenca_tempo = proximas_datas - group['Data_Evento'].values

    # Converte a diferença de tempo para minutos (float)
    minutos = diferenca_tempo / np.timedelta64(1, 'm')

    # Quem não tem falha no futuro recebe NaN (ou você pode dropar depois)
    minutos[sem_futuro] = np.nan

    # Se a própria linha for um Don't Go, também desconsideramos (vira NaN)
    # porque queremos o tempo dos erros comuns até a quebra
    # minutos[group['Is_Dont_Go'] == 1] = np.nan

    # Salva no array global
    minutos_ate_falha[indices_grupo] = minutos

# Adiciona a nova coluna na base
df_caminhao['Minutos_Ate_Proximo_Dont_Go'] = minutos_ate_falha

In [4]:
# Remove os NaNs para fazer a análise estatística limpa
distribuicao_tempo = df_caminhao['Minutos_Ate_Proximo_Dont_Go'].dropna()

print("--- Estatística Descritiva do Tempo Até o Don't Go (em Minutos) ---")
print(distribuicao_tempo.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))

--- Estatística Descritiva do Tempo Até o Don't Go (em Minutos) ---
count    3.197122e+06
mean     5.374737e+03
std      8.562040e+03
min      1.666667e-02
10%      3.085543e+01
25%      3.808015e+02
50%      1.917172e+03
75%      6.581176e+03
90%      1.482548e+04
max      7.181224e+04
Name: Minutos_Ate_Proximo_Dont_Go, dtype: float64


In [5]:
df_caminhao.dropna(subset=['Minutos_Ate_Proximo_Dont_Go'], inplace=True)

In [6]:
df_caminhao['Is_Dont_Go'].value_counts()

Is_Dont_Go
0    3177356
1      19766
Name: count, dtype: int64